# Domain and mesh sensitivity

The calculation uses asymmetric analytical depletion edges. Successive results
are compared on their common central interval. Acceptance requires both the
relative peak-field change and normalised field-profile RMS change to be below
1%.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp
from scipy.interpolate import interp1d
from scipy.optimize import least_squares
from scipy.sparse import diags

q = 1.602176634e-19
eps0 = 8.8541878128e-12
eps_si = 11.7*eps0
kB = 1.380649e-23

def depletion_edges(Na_cm3, Nd_cm3, Vbi):
    Na, Nd = Na_cm3*1e6, Nd_cm3*1e6
    d = np.sqrt(2*eps_si*Vbi/q*(Na+Nd)/(Na*Nd))
    return d, Nd*d/(Na+Nd), Na*d/(Na+Nd)

def debye_lengths(Na_cm3, Nd_cm3, T):
    Na, Nd = Na_cm3*1e6, Nd_cm3*1e6
    return (np.sqrt(eps_si*kB*T/(q*q*Na)),
            np.sqrt(eps_si*kB*T/(q*q*Nd)))

def smooth_window(x, left, right, delta):
    return 0.5*(np.tanh((x-left)/delta)-np.tanh((x-right)/delta))

def regularised_depletion(Na_cm3, Nd_cm3, T=300., ni_cm3=1e10,
                          buffer_factor=5., points=1201,
                          delta_fraction=0.01):
    Na, Nd = Na_cm3*1e6, Nd_cm3*1e6
    VT = kB*T/q
    Vbi = VT*np.log(Na_cm3*Nd_cm3/ni_cm3**2)
    d, xp, xn = depletion_edges(Na_cm3, Nd_cm3, Vbi)
    LDp, LDn = debye_lengths(Na_cm3, Nd_cm3, T)
    x = np.linspace(-xp-buffer_factor*LDp, xn+buffer_factor*LDn, points)
    delta = delta_fraction*d
    rho = q*(Nd*smooth_window(x, 0., xn, delta)
             - Na*smooth_window(x, -xp, 0., delta))
    u = x/d
    def fun(us, y):
        xs = us*d
        rs = q*(Nd*smooth_window(xs, 0., xn, delta)
                - Na*smooth_window(xs, -xp, 0., delta))
        return np.vstack((y[1], -d*d*rs/eps_si))
    def bc(ya, yb):
        return np.array([ya[0], yb[0]-Vbi])
    guess = np.vstack((Vbi*(u-u[0])/(u[-1]-u[0]),
                       np.full_like(u, Vbi/(u[-1]-u[0]))))
    sol = solve_bvp(fun, bc, u, guess, tol=2e-6, max_nodes=30000)
    if not sol.success:
        raise RuntimeError(sol.message)
    phi = sol.sol(u)[0]
    E = -sol.sol(u)[1]/d
    return dict(x=x, phi=phi, E=E, rho=rho, Vbi=Vbi, d=d, xp=xp, xn=xn,
                delta_num=delta, success=sol.success)


In [ ]:
def compare(a, b):
    left, right = max(a['x'][0], b['x'][0]), min(a['x'][-1], b['x'][-1])
    xq = np.linspace(left, right, 2001)
    Ea = interp1d(a['x'], a['E'], kind='cubic')(xq)
    Eb = interp1d(b['x'], b['E'], kind='cubic')(xq)
    peak_change = abs(np.max(abs(Eb))-np.max(abs(Ea)))/np.max(abs(Eb))
    rms = np.sqrt(np.mean((Eb-Ea)**2))/np.max(abs(Eb))
    return peak_change, rms

Na, Nd, T, ni = 1e18, 1e16, 300., 1e10  # strongly asymmetric
settings = [(5.,1201),(8.,1801),(12.,2601),(12.,5201)]
runs = [regularised_depletion(Na,Nd,T,ni,buffer_factor=b,points=n)
        for b,n in settings]
print("comparison                 peak change     profile RMS      result")
all_pass = True
for i in range(1,len(runs)):
    pc, rms = compare(runs[i-1],runs[i])
    passed = pc < .01 and rms < .01
    all_pass &= passed
    print(f"{settings[i-1]} -> {settings[i]}   {100*pc:9.4f}%   {100*rms:9.4f}%   {'PASS' if passed else 'REFINE'}")
print("FINAL:", "PASS" if all_pass else "REFINE DOMAIN OR MESH")

plt.figure(figsize=(8,4))
for r,s in zip(runs,settings):
    plt.plot(r['x']*1e6,r['E']/1e5,label=str(s))
plt.xlabel('x (um)'); plt.ylabel('E (kV/cm)'); plt.grid(); plt.legend()
plt.tight_layout(); plt.show()